# Mortgage Doc RAG — Colab Demo

End-to-end demo: dual-engine OCR → document separation → validated RAG → layered evals.

Runtime: any GPU works (T4 is fine; L4/A100 pick a larger model automatically). CPU works too, slowly.

In [ ]:
# Install (Colab)
!pip install -q uv
!git clone -q https://github.com/tccao/mortgage-doc-rag.git
%cd mortgage-doc-rag
# CUDA build of llama-cpp-python first (prebuilt wheel; falls back to ~20 min source build).
# Installed before the [llm] extra so the resolver sees it satisfied and skips the CPU sdist.
!pip install -q llama-cpp-python --index-url https://abetlen.github.io/llama-cpp-python/whl/cu124 --force-reinstall --no-deps \
 || CMAKE_ARGS="-DGGML_CUDA=on" pip install -q llama-cpp-python --force-reinstall --no-deps --no-cache-dir
!uv pip install --system -q -e . -e ".[llm]" -e ".[ui]"
!python -c "import llama_cpp; print('GPU offload available:', llama_cpp.llama_supports_gpu_offload())"

In [ ]:
# Pick model tier from detected GPU
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
else:
    gpu, vram_gb = "cpu", 0

if vram_gb >= 20:  # L4 / A100 tier
    repo, filename = "deepreinforce-ai/Ornith-1.0-35B-GGUF", "ornith-1.0-35b.Q4_K_M.gguf"
else:              # T4 tier / CPU
    repo, filename = "TheBloke/Mistral-7B-Instruct-v0.1-GGUF", "mistral-7b-instruct-v0.1.Q4_K_M.gguf"

print(f"{gpu} ({vram_gb:.0f} GB) -> {repo}")

In [ ]:
# Process the adversarial loan-file bundle
from mortgage_rag import PipelineConfig, process_files, build_orchestrator

cfg = PipelineConfig.load(llm_repo_id=repo, llm_filename=filename)
result = process_files(["data/loan_files/loan_file_01_conflicting_cd.pdf"], cfg)

print(result.stats)
print(result.consistency_report)
for f in result.files:
    for issue in f.validation_issues:
        print("!", issue)

In [ ]:
# Ask questions (note the conflicting 'corrected' closing disclosure in this bundle)
orch = build_orchestrator(result.index, cfg)

for q in ["What is the loan amount on the closing disclosure?",
          "What is the interest rate?"]:
    res = orch.answer(q)
    print(f"Q: {q}\nA: {res.answer}")
    for c in res.citations[:2]:
        print(f"   source: {c.doc_type} p{c.page_start} (score {c.score:.2f})")
    print()

In [ ]:
# Run the eval harness (retrieval layer is deterministic and fast)
!python evals/run_eval.py --retrieval-only
!head -40 evals/report.md

## Full benchmark

The committed baseline in `evals/baselines/` and the results table in the README come from
`python evals/run_eval.py --all --save-baseline` — see `docs/design.md` for the methodology
and `evals/golden_set.jsonl` for every scored case, including the adversarial bundles.

In [ ]:
# Full benchmark (long — use L4/A100). Writes evals/report.md + evals/baselines/latest.json
import os
os.environ["MRAG_LLM_REPO_ID"] = repo
os.environ["MRAG_LLM_FILENAME"] = filename

!python -u evals/run_eval.py --all --save-baseline
!cat evals/report.md